In [1]:
# Create pipeline_sla_config table and seed default SLA values

from pyspark.sql.types import (
    StructField,StructType,
    StringType,IntegerType,DoubleType,TimestampType,BooleanType
)

# DB = "yelp_lakehouse.dbo"

StatementMeta(, 21fc644d-fbe6-4149-835b-7653f2095d7f, 3, Finished, Available, Finished, False)

In [2]:
# 1. Create pipeline_sla_config
sla_config_schema = StructType([
    StructField("pipeline_name", StringType(), False),
    StructField("layer", StringType(), False),
    StructField("max_duration_sec", IntegerType(), False), #SLA: max allowed run time
    StructField("min_success_rate", DoubleType(), False), #SLA: min success rate (0-1)
    StructField("max_blocked_rate", DoubleType(), True), #SLA: max blocked rate (0-1)
    StructField("is_active", BooleanType(), True),
    StructField("update_at", TimestampType(), True)
])

(
    spark.createDataFrame([], sla_config_schema).write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("pipeline_sla_config")
    # .saveAsTable(f"{DB}.pipeline_sla_config")
)

print("pipeline_sla_config created.")


StatementMeta(, 21fc644d-fbe6-4149-835b-7653f2095d7f, 4, Finished, Available, Finished, False)

pipeline_sla_config created.


In [3]:
# 2.Seed default SLA values 

spark.sql(f"""
INSERT INTO pipeline_sla_config VALUES
('01_0_bronze_run_all', 'bronze', 600, 0.95, 0.05,1, CURRENT_TIMESTAMP()),
('02_0_silver_run_all', 'silver', 600, 0.95, 0.05, 1, CURRENT_TIMESTAMP()),
('02_2_dq_silver_run_all', 'silver', 300, 0.95, 0.10, 1, CURRENT_TIMESTAMP()),
('04_5_business_metrics', 'gold', 300, 0.95, 0.05, 1, CURRENT_TIMESTAMP()),
('04_6_gold_city_metrics', 'gold', 600, 0.95, 0.05, 1, CURRENT_TIMESTAMP())
""")

print("Default SLA values seeded.")
spark.table("pipeline_sla_config").show(truncate=False)

# spark.table(f"{DB}.pipeline_sla_config").show(truncate=False)

StatementMeta(, 21fc644d-fbe6-4149-835b-7653f2095d7f, 5, Finished, Available, Finished, False)

Default SLA values seeded.
+----------------------+------+----------------+----------------+----------------+---------+--------------------------+
|pipeline_name         |layer |max_duration_sec|min_success_rate|max_blocked_rate|is_active|update_at                 |
+----------------------+------+----------------+----------------+----------------+---------+--------------------------+
|02_2_dq_silver_run_all|silver|300             |0.95            |0.1             |true     |2026-04-18 08:56:44.480453|
|04_6_gold_city_metrics|gold  |600             |0.95            |0.05            |true     |2026-04-18 08:56:44.480453|
|04_5_business_metrics |gold  |300             |0.95            |0.05            |true     |2026-04-18 08:56:44.480453|
|01_0_bronze_run_all   |bronze|600             |0.95            |0.05            |true     |2026-04-18 08:56:44.480453|
|02_0_silver_run_all   |silver|600             |0.95            |0.05            |true     |2026-04-18 08:56:44.480453|
+------------